In [1]:
import tensorflow as tf
from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input

# 1. VGG16 Model ko ImageNet weights ke sath load karein
model = VGG16(weights='imagenet', include_top=True)

# 2. Aik 4-D Input Tensor banayein (Size: 10 images, 224x224 RGB pixels)
# VGG16 ko hamesha 224x224 size ki image chahiye hoti hai
dummy_images = tf.zeros(shape=(10, 224, 224, 3))

# 3. Tensor ko VGG16 ke mutabiq preprocess karein
preprocessed_tensor = preprocess_input(dummy_images)

# 4. Input Tensor ko VGG16 model se guzarein (Forward Pass)
output_tensor = model(preprocessed_tensor)

# 5. Output Tensor ka size aur details check karein
print("Input Tensor Shape: ", preprocessed_tensor.shape)
print("Output Tensor Shape:", output_tensor.shape)


553467096/553467096 ━━━━━━━━━━━━━━━━━━━━ 8s 0us/step
Input Tensor Shape:  (10, 224, 224, 3)
Output Tensor Shape: (10, 1000)


In [2]:
import keras
import numpy as np
from keras.applications.vgg16 import VGG16, preprocess_input, decode_predictions
from google.colab import files

# 1. VGG16 Model ko load karein
print("VGG16 Model load ho raha hai...")
model = VGG16(weights='imagenet')

# 2. Apni image computer se upload karein
print("\nApni image upload karne ke liye 'Choose Files' par click karein:")
uploaded = files.upload()

# Upload ki gayi file ka naam nikalein
img_path = list(uploaded.keys())[0]

# 3. Image ko load aur resize karein (VGG16 ko 224x224 size chahiye)
img = keras.utils.load_img(img_path, target_size=(224, 224))

# 4. Image ko numpy array mein badlein aur preprocess karein
x = keras.utils.img_to_array(img)
x = np.expand_dims(x, axis=0)  # Batch dimension add karne ke liye (1, 224, 224, 3)
x = preprocess_input(x)

# 5. Model se prediction lein
preds = model.predict(x)

# 6. Result ko aasan zaban mein print karein
print('\n--- MODEL PREDICTION RESULT ---')
# top=5 ka matlab hai top 5 sab se zyada chances wali cheezein dikhana
for imagenet_id, label, score in decode_predictions(preds, top=5)[0]:
    print(f"👉 {label}: {score * 100:.2f}%")


VGG16 Model load ho raha hai...

Apni image upload karne ke liye 'Choose Files' par click karein:


Saving dog.jpg to dog.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 793ms/step

--- MODEL PREDICTION RESULT ---
35363/35363 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
👉 golden_retriever: 82.41%
👉 Labrador_retriever: 4.67%
👉 Pembroke: 2.74%
👉 cocker_spaniel: 1.59%
👉 Cardigan: 1.59%


In [3]:
import tensorflow as tf
from tensorflow.keras.applications.vgg16 import VGG16
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Flatten, Dropout

# 1. Base VGG16 Model load karein (include_top=False ka matlab hai aakhri 1000 classes wali layers hata do)
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# 2. VGG16 ki pehle se seekhi hui layers ko freeze karein taake un ke weights kharab na hon
for layer in base_model.layers:
    layer.trainable = False

# 3. Apni custom layers lagayein jo Brain Tumor ko classification karein gi
x = Flatten()(base_model.output)            # 2D features ko 1D vector mein badlein
x = Dense(256, activation='relu')(x)       # Aik naya hidden layer patterns seekhne ke liye
x = Dropout(0.5)(x)                        # Overfitting se bachne ke liye thora gap
# Aakhri layer mein 2 neurons hain (1: Tumor, 2: No Tumor)
predictions = Dense(2, activation='softmax')(x)

# 4. Mukammal naya Medical AI Model taiyar karein
medical_model = Model(inputs=base_model.input, outputs=predictions)

# 5. Model ko compile karein
medical_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Model ki summary dekhne ke liye
medical_model.summary()


58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv1 (Conv2D)           │ (None, 224, 224, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 224, 224, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 112, 112, 128)  │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 112, 112, 128)  │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 56, 56, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 28, 28, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 28, 28, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 14, 14, 512)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 7, 7, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │     6,422,784 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │           514 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21,137,986 (80.64 MB)

 Trainable params: 6,423,298 (24.50 MB)

 Non-trainable params: 14,714,688 (56.13 MB)